### The University of Melbourne, School of Computing and Information Systems
# COMP90086 Computer Vision, 2025 Semester 2

## FINAL PROJECT

In [65]:
import tensorflow as tf
import pandas as pd
import os
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# --- Configuration ---
IMG_HEIGHT = 640
IMG_WIDTH = 480
BATCH_SIZE = 16
BASE_DIR = "Nutrition5k"
VALIDATION_SPLIT = 0.20 # 20% for validation
RANDOM_SEED = 42 # Use a seed for reproducible splits

# --- 1. Load the full labeled dataset manifest ---
full_labels_path = os.path.join(BASE_DIR, "nutrition5k_train.csv")
df = pd.read_csv(full_labels_path)
df['ID'] = df['ID'].astype(str)

print(f"DataFrame size before cleaning: {len(df)}")
df = df[df['ID'] != 'dish_2368']
print(f"DataFrame size after removing dish_2368: {len(df)}")




DataFrame size before cleaning: 3301
DataFrame size after removing dish_2368: 3300


In [66]:
# complete train and test split (with normalised pixel values)
df = df.assign(train_image_path=BASE_DIR + "\\train\\color\\" + df["ID"].astype(str) + "\\rgb.png",
               test_image_path=BASE_DIR + "\\test\\color\\" + df["ID"].astype(str) + "\\rgb.png")

datagen = ImageDataGenerator(rescale=1./255,
                             validation_split=VALIDATION_SPLIT)

train_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='training' # For training data
    )


validation_generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=os.getcwd(), # Root directory where image folders are located
        x_col='train_image_path', # Column in DataFrame containing image paths
        y_col='Value', # Column in DataFrame containing labels
        target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
        batch_size=BATCH_SIZE,
        class_mode='raw', # Or 'binary', 'sparse', 'input', None
        seed=RANDOM_SEED,
        subset='validation' # For training data
    )

Found 2640 validated image filenames.
Found 660 validated image filenames.


In [16]:
os.listdir(os.path.join(BASE_DIR, "test/color"))

['dish_3301',
 'dish_3302',
 'dish_3303',
 'dish_3304',
 'dish_3305',
 'dish_3306',
 'dish_3307',
 'dish_3308',
 'dish_3309',
 'dish_3310',
 'dish_3311',
 'dish_3312',
 'dish_3313',
 'dish_3314',
 'dish_3315',
 'dish_3316',
 'dish_3317',
 'dish_3318',
 'dish_3319',
 'dish_3320',
 'dish_3321',
 'dish_3322',
 'dish_3323',
 'dish_3324',
 'dish_3325',
 'dish_3326',
 'dish_3327',
 'dish_3328',
 'dish_3329',
 'dish_3330',
 'dish_3331',
 'dish_3332',
 'dish_3333',
 'dish_3334',
 'dish_3335',
 'dish_3336',
 'dish_3337',
 'dish_3338',
 'dish_3339',
 'dish_3340',
 'dish_3341',
 'dish_3342',
 'dish_3343',
 'dish_3344',
 'dish_3345',
 'dish_3346',
 'dish_3347',
 'dish_3348',
 'dish_3349',
 'dish_3350',
 'dish_3351',
 'dish_3352',
 'dish_3353',
 'dish_3354',
 'dish_3355',
 'dish_3356',
 'dish_3357',
 'dish_3358',
 'dish_3359',
 'dish_3360',
 'dish_3361',
 'dish_3362',
 'dish_3363',
 'dish_3364',
 'dish_3365',
 'dish_3366',
 'dish_3367',
 'dish_3368',
 'dish_3369',
 'dish_3370',
 'dish_3371',
 'dish

In [17]:
# create dataset to load test dataset
df_test = pd.DataFrame({"ID": os.listdir(os.path.join(BASE_DIR, "test/color"))})
df_test = df_test.assign(test_image_path = BASE_DIR + "\\test\\color\\" + df_test["ID"].astype(str) + "\\rgb.png")

In [18]:
df_test

,ID,test_image_path
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png
...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png


In [19]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator=test_datagen.flow_from_dataframe(
dataframe=df_test,
directory=os.getcwd(),
x_col="test_image_path",
y_col=None,
target_size=(IMG_HEIGHT, IMG_WIDTH), # Resize images to this size
batch_size=BATCH_SIZE,
seed=RANDOM_SEED,
shuffle=False,
class_mode=None)

Found 189 validated image filenames.


## improvement_1 channel stacking on using the depth image

In [67]:
# Add the path for the RGB image (as you already have)
df['rgb_path'] = df.apply(lambda row: os.path.join(BASE_DIR, 'train', 'color', str(row['ID']), 'rgb.png'), axis=1)

# ADD THIS: Add the path for the raw depth image
df['depth_path'] = df.apply(lambda row: os.path.join(BASE_DIR, 'train', 'depth_raw', str(row['ID']), 'depth_raw.png'), axis=1)

# Now your DataFrame has columns for both image paths
print(df.head())

          ID       Value                           train_image_path  \
0  dish_0000  221.167068  Nutrition5k\train\color\dish_0000\rgb.png   
1  dish_0001  140.980011  Nutrition5k\train\color\dish_0001\rgb.png   
2  dish_0002  274.335999  Nutrition5k\train\color\dish_0002\rgb.png   
3  dish_0003  589.501648  Nutrition5k\train\color\dish_0003\rgb.png   
4  dish_0004  258.599670  Nutrition5k\train\color\dish_0004\rgb.png   

                            test_image_path  \
0  Nutrition5k\test\color\dish_0000\rgb.png   
1  Nutrition5k\test\color\dish_0001\rgb.png   
2  Nutrition5k\test\color\dish_0002\rgb.png   
3  Nutrition5k\test\color\dish_0003\rgb.png   
4  Nutrition5k\test\color\dish_0004\rgb.png   

                                    rgb_path  \
0  Nutrition5k\train\color\dish_0000\rgb.png   
1  Nutrition5k\train\color\dish_0001\rgb.png   
2  Nutrition5k\train\color\dish_0002\rgb.png   
3  Nutrition5k\train\color\dish_0003\rgb.png   
4  Nutrition5k\train\color\dish_0004\rgb.png   

 

In [68]:
import cv2
import numpy as np

def custom_data_generator(dataframe, batch_size, target_size=(128, 128)):
    df = dataframe.copy()
    img_height, img_width = target_size
    
    while True:
        df = df.sample(frac=1) 
        
        for i in range(0, len(df), batch_size):
            batch_df = df.iloc[i : i + batch_size]
            
            batch_images = []
            batch_labels = []

            for index, row in batch_df.iterrows():
                # Load RGB and Depth images
                rgb_img = cv2.imread(row['rgb_path'])
                depth_img = cv2.imread(row['depth_path'], cv2.IMREAD_GRAYSCALE)

                # --- ADD THIS CHECK ---
                # Check if either image failed to load
                if rgb_img is None:
                    print(f"Error: Could not load RGB image at path: {row['rgb_path']}")
                    continue # Skip this problematic image
                
                if depth_img is None:
                    print(f"Error: Could not load depth image at path: {row['depth_path']}")
                    continue # Skip this problematic image
                # --- END OF CHECK ---

                # Now it's safe to process the images
                rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)
                rgb_resized = cv2.resize(rgb_img, (img_width, img_height)) / 255.0
                depth_resized = cv2.resize(depth_img, (img_width, img_height)) / 255.0
                
                depth_with_channel = np.expand_dims(depth_resized, axis=-1)
                stacked_image = np.concatenate([rgb_resized, depth_with_channel], axis=-1)
                
                batch_images.append(stacked_image)
                batch_labels.append(row['Value'])

            yield (np.array(batch_images), np.array(batch_labels))

In [69]:
train_df, val_df = train_test_split(
    df, 
    test_size=VALIDATION_SPLIT, # e.g., 0.20
    random_state=RANDOM_SEED    # e.g., 42, for reproducible results
)


train_gen = custom_data_generator(train_df, BATCH_SIZE, target_size=(IMG_HEIGHT, IMG_WIDTH))
val_gen = custom_data_generator(val_df, BATCH_SIZE, target_size=(IMG_HEIGHT, IMG_WIDTH))

In [70]:
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 4)),
        
        layers.Conv2D(8, (5, 5), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 636, 476, 8)    │           808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 318, 238, 8)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_7 (Flatten)             │ (None, 605472)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 10)             │     6,054,730 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,055,549 (23.10 MB)

 Trainable params: 6,055,549 (23.10 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
STEP_SIZE_TRAIN = len(train_df) // BATCH_SIZE
STEP_SIZE_VALID = len(val_df) // BATCH_SIZE

model.fit(train_gen,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=val_gen,
          validation_steps=STEP_SIZE_VALID,
          epochs=20
)


Epoch 1/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 128s 772ms/step - loss: 46372.9414 - mse: 46372.9414 - val_loss: 27467.0762 - val_mse: 27467.0762
Epoch 2/20
 48/165 ━━━━━━━━━━━━━━━━━━━━ 1:12 621ms/step - loss: 22567.1660 - mse: 22567.1660

KeyboardInterrupt: 

In [ ]:
model.evaluate(val_gen)

## Build and Train a conventional baseline classfier


In [22]:
# build basic regression model
model = tf.keras.Sequential(
    [
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        
        layers.Conv2D(8, (5, 5), activation='relu'), # fill in
        layers.MaxPooling2D((2, 2)), # fill in
        
        layers.Flatten(),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="linear")
    ], 
)

model.compile(optimizer='adam', loss='mse', metrics=['mse'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 636, 476, 8)    │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 318, 238, 8)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 605472)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │     6,054,730 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,055,349 (23.10 MB)

 Trainable params: 6,055,349 (23.10 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
STEP_SIZE_TRAIN = train_generator.n//train_generator.batch_size
STEP_SIZE_VALID = validation_generator.n//validation_generator.batch_size

model.fit(train_generator,
          steps_per_epoch=STEP_SIZE_TRAIN,
          validation_data=validation_generator,
          validation_steps=STEP_SIZE_VALID,
          epochs=20
)


C:\Users\sally\anaconda3\envs\CV\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
 7/82 ━━━━━━━━━━━━━━━━━━━━ 51s 691ms/step - loss: 60837.3047 - mse: 60837.3047

KeyboardInterrupt: 

In [ ]:
# validation
STEP_SIZE_TEST=test_generator.n//test_generator.batch_size
model.evaluate(validation_generator)

21/21 ━━━━━━━━━━━━━━━━━━━━ 7s 344ms/step - loss: 14278.6387 - mse: 14278.6387


[14662.9150390625, 14662.9150390625]

In [21]:
test_generator.reset()
preds=model.predict(test_generator)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 376ms/step


In [22]:
df_test = df_test.assign(Value=preds)

In [23]:
df_test

,ID,test_image_path,Value
0,dish_3301,Nutrition5k\test\color\dish_3301\rgb.png,853.766052
1,dish_3302,Nutrition5k\test\color\dish_3302\rgb.png,152.479446
2,dish_3303,Nutrition5k\test\color\dish_3303\rgb.png,91.166473
3,dish_3304,Nutrition5k\test\color\dish_3304\rgb.png,161.127014
4,dish_3305,Nutrition5k\test\color\dish_3305\rgb.png,360.575531
...,...,...,...
184,dish_3485,Nutrition5k\test\color\dish_3485\rgb.png,132.760986
185,dish_3486,Nutrition5k\test\color\dish_3486\rgb.png,0.033012
186,dish_3487,Nutrition5k\test\color\dish_3487\rgb.png,378.267456
187,dish_3488,Nutrition5k\test\color\dish_3488\rgb.png,165.710480


In [24]:
df_submit = df_test.drop("test_image_path", axis=1)

In [25]:
df_submit

,ID,Value
0,dish_3301,853.766052
1,dish_3302,152.479446
2,dish_3303,91.166473
3,dish_3304,161.127014
4,dish_3305,360.575531
...,...,...
184,dish_3485,132.760986
185,dish_3486,0.033012
186,dish_3487,378.267456
187,dish_3488,165.710480


In [26]:
# save copy to csv
df_submit.to_csv("baseline_submission.csv")